### Tier 2 Recommendation Engine

This notebook trains the second tier of the AyurFit Machine Learning pipeline. It maps the predicted Disease and Symptom Severity to specific Ayurvedic Herbs using a transparent Decision Tree algorithm.

In [2]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib

### Load Data

We load the curated dataset and explicitly drop any records missing critical information required for the Tier 2 mapping: `Disease`, `Symptom Severity`, or `Ayurvedic Herbs`.

In [3]:
df = pd.read_csv('../dataset/final ayurfit.csv')
df = df.dropna(subset=['Disease', 'Symptom Severity', 'Ayurvedic Herbs'])

### Encode Features and Target

We reuse the exact `disease_label_encoder.joblib` from Tier 1 to maintain architectural consistency. We also map the ordinal text of `Symptom Severity` to numerical weights, and encode the distinct `Ayurvedic Herbs` combinations as categorical targets.

In [4]:
# Load Tier 1 Disease Encoder
disease_encoder = joblib.load('disease_label_encoder.joblib')

# Ensure dataset only includes diseases the encoder knows
known_diseases = set(disease_encoder.classes_)
df = df[df['Disease'].isin(known_diseases)]
df['Disease_Encoded'] = disease_encoder.transform(df['Disease'])

# Map Symptom Severity text to integer weights
severity_map = {
    'Low': 0,
    'Mild to Moderate': 1,
    'Moderate': 2,
    'Moderate to High': 3,
    'Moderate to Severe': 4,
    'Mild to Severe': 5,
    'High': 6,
    'Severe': 7
}
df['Severity_Encoded'] = df['Symptom Severity'].map(severity_map).fillna(1).astype(int)

# Encode the target Ayurvedic Herbs
herbs_encoder = LabelEncoder()
df['Herbs_Encoded'] = herbs_encoder.fit_transform(df['Ayurvedic Herbs'])

### Train/Test Split

We isolate our input feature matrix `X` (Disease and Severity) and our target matrix `y` (Herbs), splitting 80% for training and 20% for evaluation.

In [5]:
X = df[['Disease_Encoded', 'Severity_Encoded']]
y = df['Herbs_Encoded']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Train Decision Tree

Using a Decision Tree ensures explicit, explainable clinical rules for remedy mapping. It naturally captures the logical hierarchy between the primary disease and its severity modifier.

In [6]:
dt_classifier = DecisionTreeClassifier(random_state=42)
dt_classifier.fit(X_train, y_train)

train_acc = accuracy_score(y_train, dt_classifier.predict(X_train))
test_acc = accuracy_score(y_test, dt_classifier.predict(X_test))

print(f"Training Accuracy: {train_acc * 100:.2f}%")
print(f"Testing Accuracy: {test_acc * 100:.2f}%")

Training Accuracy: 93.52%
Testing Accuracy: 91.50%


### Export Models

We export the trained Tier 2 mapping model and the new `herbs_label_encoder.joblib` for the production backend.

In [7]:
joblib.dump(dt_classifier, 'tier2_recommender.joblib')
joblib.dump(herbs_encoder, 'herbs_label_encoder.joblib')
print("Tier 2 Models successfully exported!")

Tier 2 Models successfully exported!
